## Analysis of the PREACT-digital study sample

In [8]:
from pathlib import Path
import sys
from datetime import date
import pandas as pd
import gc
import os
import glob
import numpy as np
import pickle

# --- Paths / imports -------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
PREPROCESSING_DIR = PROJECT_ROOT / "functions" / "preprocessing"
for p in (PROJECT_ROOT, PREPROCESSING_DIR):
    if str(p) not in sys.path:
        sys.path.append(str(p))

from server_config import (
    datapath,
    proj_sheet,
    preprocessed_path, redcap_path,
    raw_path,
    backup_path,
)

In [9]:
with open(preprocessed_path + '/monitoring_data.pkl', 'rb') as file:
    df_monitoring = pickle.load(file)
    
with open(redcap_path + '/redcap_data.pkl', 'rb') as file:
    df_redcap = pickle.load(file)

split_path = redcap_path + "/111025_subject_splits.csv"
df_split = pd.read_csv(split_path)

### Check study timeframes

In [10]:

t5_path_zert = redcap_path + "/t5" + "/Zertifizierung_FOR_Verlaufsdoku_T5.csv"
t5_path = redcap_path + "/t5" + "/FOR_Verlaufsdoku_T5.csv"

t5_zert = pd.read_csv(t5_path_zert, low_memory="False")
t5 = pd.read_csv(t5_path, low_memory="False")

In [11]:
cols_to_keep = ['for_id','t5_session_date', 't1_session_date','t20_session_date','t5_assessment_date','t20_assessment_date','post_assessment_date', 'ic_date']

In [12]:
t5_short = t5[cols_to_keep]
t5_zert_short = t5_zert[cols_to_keep]
t5_final = pd.concat([t5_zert_short, t5_short], ignore_index=True)

In [13]:
t5_final = t5_final.drop_duplicates()

In [14]:
# (optional) ensure it's a datetime so sorting behaves correctly
t5_final['t5_assessment_date'] = pd.to_datetime(t5_final['t5_assessment_date'], errors='coerce')

dedup = (
    t5_final.sort_values(['for_id', 't5_assessment_date'], na_position='first')  # NaNs first
      .drop_duplicates(subset='for_id', keep='last')                       # keep latest non-null
      .reset_index(drop=True)
)


In [15]:
dedup

,for_id,t5_session_date,t1_session_date,t20_session_date,t5_assessment_date,t20_assessment_date,post_assessment_date,ic_date
0,FOR11001,2023-07-28,2023-07-06,2023-10-24,2023-07-31,2023-10-26,2024-10-01,2023-05-24
1,FOR11002,2023-08-15,2023-08-03,2023-11-14,2023-08-17,2023-11-14,2024-08-20,2023-05-24
2,FOR11003,2023-10-24,2023-09-19,2024-03-12,2023-10-29,2024-03-18,NaN,2023-05-31
3,FOR11004,2023-10-12,2023-09-14,2024-01-25,2023-10-18,2024-01-25,2024-10-11,2023-05-31
4,FOR11005,2023-08-22,2023-08-10,2024-02-01,2023-08-31,2024-02-05,NaN,2023-06-06
...,...,...,...,...,...,...,...,...
584,FOR14190,NaN,NaN,NaN,NaT,NaN,NaN,NaN
585,FOR14191,NaN,NaN,NaN,NaT,NaN,NaN,NaN
586,FOR14901,2023-10-05,2023-08-17,NaN,2023-09-28,NaN,2023-12-19,2023-06-08
587,FOR14902,2023-07-25,2023-06-30,2024-01-26,2023-07-25,2024-01-26,NaN,2023-06-09
